In [1]:
import sys
import subprocess
import importlib.util

AUTO_INSTALL_MISSING_PACKAGES = True

required_packages = {
    "drms": "drms",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "sunpy": "sunpy",
    "zeep": "zeep"
}

for package_name, import_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
    else:
        print(f"Package already available: {package_name}")
from sunpy.net import Fido
from sunpy.net import attrs as a
import drms
import random
import numpy as np
from datetime import datetime, timedelta
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.feature_selection import SelectKBest, f_classif

Installing missing package: drms
Package already available: pandas
Package already available: numpy
Package already available: matplotlib
Package already available: tqdm
Installing missing package: sunpy
Installing missing package: zeep


In [5]:
#Queries registed flare events for SHARP data

event_type = "FL"
tstart = "2010/5/1"
tend = "2014/5/1"
result = Fido.search(a.Time(tstart, tend),
                     a.hek.EventType(event_type),
                     a.hek.FL.GOESCls > "M1.0",
                     a.hek.OBS.Observatory == "GOES")
hek_results = result["hek"]
filtered_results = hek_results["event_starttime", "event_peaktime",
                               "event_endtime", "fl_goescls", "ar_noaanum","hrc_coord","hpc_coord"]
print(len(filtered_results))


186


In [3]:
#Queries SHARP data that corresponds to 24 hours prior to flare event

client = drms.Client()
sharp_data_list = []

for flare in filtered_results:
    noaa_num = flare["ar_noaanum"]
    flare_time_str = str(flare["event_starttime"])


    if not noaa_num or noaa_num == 0 or str(noaa_num).upper() == "NONE":
        continue
    flare_dt = datetime.fromisoformat(flare_time_str.replace("Z", ""))
    #Can adjust the prediction window to 12 hours or 6 hours can changing the 24
    prior_24h_dt = flare_dt - timedelta(hours=24)

    jsoc_time_str = prior_24h_dt.strftime("%Y.%m.%d_%H:%M:%S_TAI")

    try:
        keys = client.query(
            f"hmi.sharp_720s[][{jsoc_time_str}]",
            key="HARPNUM, NOAA_ARS, T_REC, USFLUX,TOTBSQ,TOTPOT,TOTUSJZ,SAVNCPP,AREA_ACR,TOTFZ,MEANPOT,R_VALUE,EPSZ,SHRGT45,MEANSHR,MEANGAM,MEANGBT,MEANGBZ,MEANGBH",
        )

        if not keys.empty:
            matched_sharp = keys[
                keys["NOAA_ARS"].astype(str).str.contains(str(noaa_num))
            ]

            if not matched_sharp.empty:
                # Add flare metadata alongside the SHARP metrics
                record = matched_sharp.iloc[0].to_dict()
                record["FLARE_LABEL"] = 1
                record["FLARE_NOAA_NUM"] = noaa_num
                record["FLARE_STARTTIME"] = flare_time_str
                record["FLARE_CLASS"] = flare["fl_goescls"]

                sharp_data_list.append(record)

    except Exception as e:
        print(f"Error querying NOAA {noaa_num} at {jsoc_time_str}: {e}")

# Convert results to DataFrame
found_sharp_data = pd.DataFrame(sharp_data_list)
print(f"Successfully matched {len(found_sharp_data)} / {len(filtered_results)} flares to SHARP data.")
sharp_positive_classes = found_sharp_data.sample(n=200, random_state=42)
# print(positive_classes.head())

Successfully matched 68 / 73 flares to SHARP data.


ValueError: Cannot take a larger sample than population when 'replace=False'

In [ ]:
#Queries random flare events that don'y overlap with flare events

buffer_hrs = 24
flare_exclusion_windows = []

for flare in hek_results:
    start_str = str(flare["event_starttime"]).replace("Z", "").replace(" ", "T")
    end_str = str(flare["event_endtime"]).replace("Z", "").replace(" ", "T")

    t_start = datetime.fromisoformat(start_str)
    t_end = datetime.fromisoformat(end_str)

    # Expand interval by 24 hours on both sides
    excl_start = t_start - timedelta(hours=buffer_hrs)
    excl_end = t_end + timedelta(hours=buffer_hrs)
    #adds all flare into one giant exclusion window
    flare_exclusion_windows.append((excl_start, excl_end))

print(f"Loaded {len(flare_exclusion_windows)} flare exclusion windows.")

def is_near_flare(candidate_dt):
    for window_start, window_end in flare_exclusion_windows:
        if window_start <= candidate_dt <= window_end:
            return True
    return False

client = drms.Client()
non_flare_samples = []

start_date = datetime(2010, 5, 1)
end_date = datetime(2014, 5, 1)
total_seconds = int((end_date - start_date).total_seconds())

#Defaults to 500 non-flare events but can change to adjust class ratio
while len(non_flare_samples) < 500:
    random_sec = random.randint(0, total_seconds)
    candidate_dt = start_date + timedelta(seconds=random_sec)
    #Makes sure selected time doesn't coincide with flare event
    if is_near_flare(candidate_dt):
        continue
    jsoc_time_str = candidate_dt.strftime("%Y.%m.%d_%H:%M:%S_TAI")

    try:
        keys = client.query(
            f"hmi.sharp_720s[][{jsoc_time_str}]",
            key="HARPNUM, NOAA_ARS, T_REC, USFLUX,TOTBSQ,TOTPOT,TOTUSJZ,SAVNCPP,AREA_ACR,TOTFZ,MEANPOT,R_VALUE,EPSZ,SHRGT45,MEANSHR,MEANGAM,MEANGBT,MEANGBZ",
        )

        if not keys.empty:
            sample = keys.sample(n=1).iloc[0].to_dict()
            sample["FLARE_LABEL"] = 0  # Non-flare label
            sample["QUERY_TIMESTAMP"] = jsoc_time_str

            non_flare_samples.append(sample)

    except Exception as e:
        continue

sharp_negative_classes = pd.DataFrame(non_flare_samples)
print("\nSampling Complete!")

In [ ]:
#Queries registed flare events for SHARP data

event_type = "FL"
tstart = "2000/1/1"
tend = "2004/1/1"
result = Fido.search(a.Time(tstart, tend),
                     a.hek.EventType(event_type),
                     a.hek.FL.GOESCls > "M5.0",
                     a.hek.OBS.Observatory == "GOES")
hek_results = result["hek"]
filtered_results = hek_results["event_starttime", "event_peaktime",
                               "event_endtime", "fl_goescls", "ar_noaanum","hrc_coord","hpc_coord"]
print(len(filtered_results))


In [ ]:
#Queries SMARP data that corresponds to 24 hours prior to flare event

client = drms.Client()
smarp_data_list = []

for flare in filtered_results:
    noaa_num = flare["ar_noaanum"]
    flare_time_str = str(flare["event_starttime"])


    if not noaa_num or noaa_num == 0 or str(noaa_num).upper() == "NONE":
        continue
    flare_dt = datetime.fromisoformat(flare_time_str.replace("Z", ""))
    #Can adjust the prediction window to 12 hours or 6 hours can changing the 24
    prior_24h_dt = flare_dt - timedelta(hours=24)

    jsoc_time_str = prior_24h_dt.strftime("%Y.%m.%d_%H:%M:%S_TAI")

    try:
        keys = client.query(
            f"mdi.smarp_96m[][{jsoc_time_str}/96m]",
            key="TARPNUM, NOAA_ARS,T_REC,AREA_SPT,AREA_ACR,NPIX_SPT,USFLUXL,R_VALUE,SIZE_ACR,AREA,MTOT,MNEG_TOT,MPOS_TOT,MSTDEV,NPIX"
        )

        if not keys.empty:
            matched_sharp = keys[
                keys["NOAA_ARS"].astype(str).str.contains(str(noaa_num))
            ]

            if not matched_sharp.empty:
                # Add flare metadata alongside the SMARP metrics
                record = matched_sharp.iloc[0].to_dict()
                record["FLARE_LABEL"] = 1
                record["FLARE_NOAA_NUM"] = noaa_num
                record["FLARE_STARTTIME"] = flare_time_str
                record["FLARE_CLASS"] = flare["fl_goescls"]

                smarp_data_list.append(record)

    except Exception as e:
        print(f"Error querying NOAA {noaa_num} at {jsoc_time_str}: {e}")

# Convert results to DataFrame
found_smarp_data = pd.DataFrame(smarp_data_list)
print(f"Successfully matched {len(found_smarp_data)} / {len(filtered_results)} flares to SMARP data.")
smarp_positive_classes = found_smarp_data.sample(n=200, random_state=42)
# print(positive_classes.head())

In [ ]:
#Queries random flare events that don'y overlap with flare events

buffer_hrs = 24
flare_exclusion_windows = []

for flare in hek_results:
    start_str = str(flare["event_starttime"]).replace("Z", "").replace(" ", "T")
    end_str = str(flare["event_endtime"]).replace("Z", "").replace(" ", "T")

    t_start = datetime.fromisoformat(start_str)
    t_end = datetime.fromisoformat(end_str)

    # Expand interval by 24 hours on both sides
    excl_start = t_start - timedelta(hours=buffer_hrs)
    excl_end = t_end + timedelta(hours=buffer_hrs)

    flare_exclusion_windows.append((excl_start, excl_end))

print(f"Loaded {len(flare_exclusion_windows)} flare exclusion windows.")

def is_near_flare(candidate_dt):
    for window_start, window_end in flare_exclusion_windows:
        if window_start <= candidate_dt <= window_end:
            return True
    return False

client = drms.Client()
non_flare_samples = []

start_date = datetime(2000,1, 1)
end_date = datetime(2004, 1, 1)
total_seconds = int((end_date - start_date).total_seconds())

#Defaults to 500 non-flare events but can change to adjust class ratio
while len(non_flare_samples) < 500:
    random_sec = random.randint(0, total_seconds)
    candidate_dt = start_date + timedelta(seconds=random_sec)
    #Makes sure selected time doesn't coincide with flare event
    if is_near_flare(candidate_dt):
        continue
    jsoc_time_str = candidate_dt.strftime("%Y.%m.%d_%H:%M:%S_TAI")

    try:
        keys = client.query(
            f"mdi.smarp_96m[][{jsoc_time_str}/96m]",
            key="TARPNUM, NOAA_ARS,T_REC,AREA_SPT,AREA_ACR,NPIX_SPT,USFLUXL,R_VALUE,SIZE_ACR,AREA,MTOT,MNEG_TOT,MPOS_TOT,MSTDEV,NPIX"
        )

        if not keys.empty:
            sample = keys.sample(n=1).iloc[0].to_dict()
            sample["FLARE_LABEL"] = 0  # Non-flare label
            sample["QUERY_TIMESTAMP"] = jsoc_time_str

            non_flare_samples.append(sample)

    except Exception as e:
        continue

# --- Step 4: Convert to DataFrame ---
smarp_negative_classes = pd.DataFrame(non_flare_samples)
print("\nSampling Complete!")

In [58]:

#Clean individual feature matrix function (Leakage-Safe)
def prepare_train_test(X_train, X_test):
    # Convert to numeric & replace infinite values
    X_train = X_train.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    X_test = X_test.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

    # Drop constant columns based only on training data
    valid_cols = X_train.columns[X_train.nunique() > 1]
    X_train = X_train[valid_cols]
    X_test = X_test[valid_cols]

    # Fill NaNs in both sets using the training set median
    train_medians = X_train.median()
    X_train = X_train.fillna(train_medians)
    X_test = X_test.fillna(train_medians)

    return X_train, X_test


# Build clean datasets separately to misalignment
sharp_feature_cols = ["USFLUX","TOTBSQ","TOTPOT","TOTUSJZ","SAVNCPP","AREA_ACR","TOTFZ","MEANPOT","R_VALUE","EPSZ","SHRGT45","MEANSHR","MEANGAM","MEANGBT","MEANGBZ"]
smarp_feature_cols = ["TARPNUM", "NOAA_ARS","T_REC","AREA_SPT","AREA_ACR","NPIX_SPT","USFLUXL","R_VALUE","SIZE_ACR","AREA","MTOT","MNEG_TOT","MPOS_TOT","MSTDEV","NPIX"]
sharp_dataset = pd.concat([sharp_positive_classes, sharp_negative_classes.sample(n=300, random_state=42)], ignore_index=True)
sharp_dataset = sharp_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# Drop any accidental missing target labels
sharp_dataset = sharp_dataset.dropna(subset=["FLARE_LABEL"])

X_sharp = sharp_dataset[sharp_feature_cols]
y_sharp = sharp_dataset["FLARE_LABEL"].astype(int)

X_train_sharp_raw, X_test_sharp_raw, y_train_sharp, y_test_sharp = train_test_split(
    X_sharp, y_sharp, test_size=0.2, random_state=42, stratify=y_sharp
)
X_train_sharp, X_test_sharp = prepare_train_test(X_train_sharp_raw, X_test_sharp_raw)


smarp_dataset = pd.concat([smarp_positive_classes, smarp_negative_classes.sample(n=300, random_state=42)], ignore_index=True)
smarp_dataset = smarp_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# Drop any accidental missing target labels
smarp_dataset = smarp_dataset.dropna(subset=["FLARE_LABEL"])

smarp_numeric_cols = [c for c in smarp_feature_cols if c not in ["TARPNUM", "NOAA_ARS", "T_REC"]]

X_smarp = smarp_dataset[smarp_numeric_cols]
y_smarp = smarp_dataset["FLARE_LABEL"].astype(int)

#Split data into training (80%) and testing (20%)
X_train_smarp_raw, X_test_smarp_raw, y_train_smarp, y_test_smarp = train_test_split(
    X_smarp, y_smarp, test_size=0.2, random_state=42, stratify=y_smarp
)
X_train_smarp, X_test_smarp = prepare_train_test(X_train_smarp_raw, X_test_smarp_raw)

print(f"SHARP Train/Test shapes: {X_train_sharp.shape}, {X_test_sharp.shape}")
print(f"SMARP Train/Test shapes: {X_train_smarp.shape}, {X_test_smarp.shape}")

SHARP Train/Test shapes: (400, 12), (100, 12)
SMARP Train/Test shapes: (400, 12), (100, 12)


In [59]:
# SHARP random forest
sharp_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)
sharp_rf.fit(X_train_sharp, y_train_sharp)


y_probs = sharp_rf.predict_proba(X_test_sharp)[:, 1]

#Can change value to adjust probability threshold
threshold = 0.50
y_pred = (y_probs >= threshold).astype(int)

# Evaluate metrics using SHARP test set
accuracy = accuracy_score(y_test_sharp, y_pred)
print("Accuracy:", accuracy)

cm = confusion_matrix(y_test_sharp, y_pred)
tn, fp, fn, tp = cm.ravel()

print("\nConfusion Matrix:")
print(cm)

# Calculate True Skill Statistic (TSS)
tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
tss = tpr - fpr

print(f"\nTrue Positive Rate (TPR / Recall): {tpr:.4f}")
print(f"False Positive Rate (FPR):         {fpr:.4f}")
print(f"True Skill Statistic (TSS):        {tss:.4f}")

Accuracy: 0.87

Confusion Matrix:
[[55  5]
 [ 8 32]]

True Positive Rate (TPR / Recall): 0.8000
False Positive Rate (FPR):         0.0833
True Skill Statistic (TSS):        0.7167


In [56]:
# SMARP random forest
smarp_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)
smarp_rf.fit(X_train_smarp, y_train_smarp)

y_probs = smarp_rf.predict_proba(X_test_smarp)[:, 1]

#Can change value to adjust probability threshold
threshold = 0.50
y_pred_smarp = (y_probs >= threshold).astype(int)

# Evaluate metrics using SMARP test set
accuracy_smarp = accuracy_score(y_test_smarp, y_pred_smarp)
print("SMARP Accuracy:", accuracy_smarp)

cm_smarp = confusion_matrix(y_test_smarp, y_pred_smarp)
tn, fp, fn, tp = cm_smarp.ravel()

print("\nSMARP Confusion Matrix:")
print(cm_smarp)

# Calculate True Skill Statistic (TSS) for SMARP
tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
tss = tpr - fpr

print(f"\nTrue Positive Rate (TPR / Recall): {tpr:.4f}")
print(f"False Positive Rate (FPR):         {fpr:.4f}")
print(f"True Skill Statistic (TSS):        {tss:.4f}")

SMARP Accuracy: 0.88

SMARP Confusion Matrix:
[[53  7]
 [ 5 35]]

True Positive Rate (TPR / Recall): 0.8750
False Positive Rate (FPR):         0.1167
True Skill Statistic (TSS):        0.7583
